# Topic05 — Which request gets the next token?

Use the same support-ticket classifier as Topic04. Short ticket: “I forgot my password and cannot sign in to my account.” Gold: access. Competing ticket: “I can log in, but clicking Export crashes the application.” Gold: bug. Gold labels are not promised model outputs.

First account for KV positions and physical blocks. Then implement arrival-aware slot admission in a decode-only toy. Finally specify a matched moderate-load pipeline/vLLM comparison. CPU ticks are invented units, never GPU seconds. The CPU mechanisms and supplied real-model comparison passed source-bound instructor rehearsal; this does not establish independent student learning. Do not submit CPU results as GPU completion. Student pilot and judge are separate.


## Reproducible execution environment

The Colab kernel forwards each visible `%%lsm` cell to one managed Python
3.11.17 worker. Scientific imports and your functions live in that worker,
not in Colab's provider namespace. Keep `%%lsm` as the first line when adding
a scientific cell. Errors remain visible; there is no automatic retry.
This isolates dependencies, not the OS, driver or untrusted code.

First verify the supplied inputs, then install the frozen CPU profile, then
start the worker. CPU setup installs no Torch, vLLM or bitsandbytes. GPU
installation remains at the topic's explicit GPU stage. Free GPU quota is
not guaranteed.


Topic05 exact code passed two full live-browser instructor runs on 13 October 2026, T4 / standard RAM. Learner validation and free GPU quota remain unverified; results and solutions stay private.


In [ ]:
#@title Provided frozen runtime inputs { display-mode: "form" }
import base64, hashlib, io, json, zipfile, uuid
from pathlib import Path
if globals().get('bridge') is not None and bridge.process.poll() is None:
    raise RuntimeError('Owned worker is still alive; preserve results and close it first')
RUNTIME_ROOT = Path('/content/lsm-notebook-runtime-'+uuid.uuid4().hex)
assert not RUNTIME_ROOT.exists(), 'Fresh attempt required; earlier evidence is preserved'
RUNTIME_ROOT.mkdir()
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == '64a304935e0a20428d8d6a717d49f021bb64e1a91a4856d4e58b0eb0bcc76e36'
expected = {'runtime.py': '097b5f36ccaa602bf0263cd0a2e875aa9a7081c40e6657607149961b1f68f8fa', 'bridge.py': '43895d5a988621f46f124d256a548d46ac07b134858a90acd9b192cb0b615826', 'worker.py': '6cb5061f1275cf672c58ebf786ddac77bb22b29fed4f8b705e2cae7e073accfe', 'pyproject.toml': '189a229da208f06e0b2675db84376bc50710fe066602a017d8dedde4bc3559bc', 'uv.lock': 'b7afd87078d41bff9753d5c54e7e7d58326baeafb93641ee3949e0ee799c3885'}
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    assert set(archive.namelist()) == set(expected) | {'manifest.json'}
    for name in archive.namelist():
        assert '/' not in name and '\\' not in name and name not in ('.','..')
        data = archive.read(name)
        if name in expected:
            assert hashlib.sha256(data).hexdigest() == expected[name], name
        with (RUNTIME_ROOT/name).open('xb') as destination:
            destination.write(data)
print('Runtime inputs verified; no packages or models installed')


### Install the frozen CPU profile

The report names the actual managed executable/version/hash and import paths. Installer logs remain in the fresh attempt directory. No base scientific packages are inherited.


Topic05 exact code passed two full live-browser instructor runs on 13 October 2026, T4 / standard RAM. Learner validation and free GPU quota remain unverified; results and solutions stay private.


In [ ]:
import importlib.util
runtime_spec = importlib.util.spec_from_file_location('lsm_runtime',RUNTIME_ROOT/'runtime.py')
runtime = importlib.util.module_from_spec(runtime_spec)
runtime_spec.loader.exec_module(runtime)
RUNTIME_PYTHON = runtime.install_runtime(RUNTIME_ROOT,'cpu')
print(json.dumps(json.loads((RUNTIME_ROOT/'identity-cpu.json').read_text()),indent=2))

### Start one persistent scientific namespace

The following provided cell registers the literal-code transport. Existing variables remain available across `%%lsm` cells; the provider and worker namespaces are separate.


Topic05 exact code passed two full live-browser instructor runs on 13 October 2026, T4 / standard RAM. Learner validation and free GPU quota remain unverified; results and solutions stay private.


In [ ]:
bridge_spec = importlib.util.spec_from_file_location('lsm_bridge',RUNTIME_ROOT/'bridge.py')
bridge_module = importlib.util.module_from_spec(bridge_spec)
bridge_spec.loader.exec_module(bridge_module)
bridge = bridge_module.IsolatedCellBridge(RUNTIME_PYTHON,RUNTIME_ROOT/'worker.py')
def run_isolated(line: str, cell: str) -> None:
    """Execute literal cell code in the owned managed worker.

    Parameters
    ----------
    line : str
        Magic arguments; empty for this transport.
    cell : str
        Actual visible Python source, not a precomputed output.

    Returns
    -------
    None
        Renders actual output and propagates failures.
    """
    if line.strip():
        raise ValueError('This transport accepts no magic arguments')
    bridge.render(cell)
get_ipython().register_magic_function(run_isolated,'cell','lsm')
print('One managed worker registered; no GPU package imported')

In [ ]:
%%lsm
#@title Provided readable CPU helper; no inference { display-mode: "form" }
from pathlib import Path
import hashlib, importlib.util, sys
HELPER_SOURCE = '"""Provided CPU models for Topic05; no inference, timing claims or student answers."""\nimport math\nfrom typing import Callable, Sequence, TypedDict\n\n\nclass KVAccount(TypedDict):\n    """Independent request payload/capacity counts; excludes shared blocks and metadata."""\n    blocks: int\n    live_positions: int\n    capacity_positions: int\n    unused_positions: int\n    payload_bytes: int\n    capacity_bytes: int\n\n\nclass Request(TypedDict):\n    """Invented request: integer arrival tick and positive decode iterations, not seconds."""\n    id: str\n    arrival: int\n    decode_steps: int\n\n\nclass Tick(TypedDict):\n    """Work during [tick,tick+1); completion/release occurs at tick+1."""\n    tick: int\n    active: list[str]\n    waiting: list[str]\n    finished: list[str]\n\n\nclass Schedule(TypedDict):\n    """Decode-only deterministic toy: trace, completion ticks and all-attempt residence."""\n    mode: str\n    ticks: list[Tick]\n    completion: dict[str, int]\n    residence_ticks: dict[str, int]\n    makespan_ticks: int\n    idle_slot_ticks: int\n\n\nREQUESTS: list[Request] = [\n    {\'id\':\'A\',\'arrival\':0,\'decode_steps\':5},\n    {\'id\':\'B\',\'arrival\':0,\'decode_steps\':1},\n    {\'id\':\'C\',\'arrival\':0,\'decode_steps\':1},\n    {\'id\':\'D\',\'arrival\':2,\'decode_steps\':2},\n]\nQWEN = {\'layers\':24, \'kv_heads\':2, \'head_width\':64, \'element_bytes\':2}\n\n\ndef require_count(value: int, name: str, minimum: int = 0) -> None:\n    """Validate one integer without accepting booleans.\n\n    Parameters\n    ----------\n    value : int\n        Count, never a probability or float.\n    name : str\n        Diagnostic field name.\n    minimum : int, default 0\n        Inclusive lower bound.\n\n    Returns\n    -------\n    None\n        Raises ValueError for a noninteger or a count below the bound.\n    """\n    if type(value) is not int or value < minimum:\n        raise ValueError(name + \' must be an integer >= \' + str(minimum))\n\n\ndef kv_bytes_per_position(layers: int, kv_heads: int, head_width: int,\n                          element_bytes: int) -> int:\n    """Compute keys AND values across all layers for one retained position.\n\n    Parameters\n    ----------\n    layers : int\n        Positive number of full-attention layers.\n    kv_heads : int\n        Positive KV head count, not query heads. No tensor parallelism.\n    head_width : int\n        Positive scalar elements per KV head.\n    element_bytes : int\n        Positive bytes per scalar; 2 for this FP16 toy.\n\n    Returns\n    -------\n    int\n        Payload bytes per position; excludes scales, block tables and alignment.\n\n    Raises\n    ------\n    ValueError\n        Invalid dimensions; boolean integers are rejected.\n    """\n    for name,value in [(\'layers\',layers),(\'kv_heads\',kv_heads),\n                       (\'head_width\',head_width),(\'element_bytes\',element_bytes)]:\n        require_count(value,name,1)\n    return 2*layers*kv_heads*head_width*element_bytes\n\n\ndef check_account(function: Callable[[Sequence[int], int, int], KVAccount]) -> int:\n    """Check the student\'s independent-per-request block accounting.\n\n    Parameters\n    ----------\n    function : Callable\n        account_kv(lengths, block_size, bytes_per_position). Zero-length requests\n        have no block. Sum the independently rounded capacities; do not mutate inputs.\n\n    Returns\n    -------\n    int\n        Number of fixture checks; success is arithmetic, not GPU evidence.\n    """\n    cases=[([],16,12,0,0),([0],16,12,0,0),([16],16,12,1,16),\n           ([17],16,12,2,17),([17,31,33],16,12288,7,81),\n           ([1,1],16,12,2,2),([15,16,17],8,12,7,48)]\n    for lengths,size,bpp,blocks,live in cases:\n        before=list(lengths)\n        actual=function(lengths,size,bpp)\n        assert actual==dict(blocks=blocks,live_positions=live,\n            capacity_positions=blocks*size,unused_positions=blocks*size-live,\n            payload_bytes=live*bpp,capacity_bytes=blocks*size*bpp),(lengths,actual)\n        assert lengths==before,\'Do not mutate request lengths\'\n    invalid=[([-1],16,12),([True],16,12),([1.5],16,12),([1],0,12),\n             ([1],True,12),([1],16,0),([1],16,True)]\n    for args in invalid:\n        try: function(*args)\n        except ValueError: continue\n        raise AssertionError(\'Invalid count accepted: \'+repr(args))\n    return len(cases)+len(invalid)\n\n\ndef check_admission(function: Callable[[Sequence[Request], int, int], list[str]]) -> int:\n    """Check arrival-aware FIFO selection from an already ordered pending queue.\n\n    Parameters\n    ----------\n    function : Callable\n        admit_fifo(pending,tick,free_slots); returns IDs of arrived requests only,\n        preserving queue order. This does not model prefill or memory admission.\n\n    Returns\n    -------\n    int\n        Fixture count. Failing IDs/order/count or invalid inputs raise an error.\n    """\n    cases=[(REQUESTS,0,2,[\'A\',\'B\']),(REQUESTS[2:],1,2,[\'C\']),\n           (REQUESTS[2:],2,2,[\'C\',\'D\']),(REQUESTS,0,0,[]),([],0,2,[])]\n    for pending,tick,slots,want in cases:\n        before=[dict(r) for r in pending]\n        assert function(pending,tick,slots)==want\n        assert pending==before,\'Selection must not mutate the queue\'\n    for tick,slots in [(-1,1),(0,-1),(True,1),(0,True),(0,1.5)]:\n        try: function(REQUESTS,tick,slots)\n        except ValueError: continue\n        raise AssertionError(\'Invalid tick/slots accepted\')\n    return len(cases)+5\n\n\ndef simulate(admit: Callable[[Sequence[Request], int, int], list[str]],\n             requests: Sequence[Request], slots: int = 2,\n             mode: str = \'continuous\') -> Schedule:\n    """Compare iteration-refill with one static cohort at a time.\n\n    Parameters\n    ----------\n    admit : Callable\n        Student arrival-aware FIFO selector. Called at every eligible boundary.\n    requests : Sequence[Request]\n        Unique nonempty IDs in FIFO order, nonnegative arrival ticks and positive\n        decode work. Known lengths simulate stop events, not a real length oracle.\n    slots : int, default 2\n        Positive concurrent decode capacity. No prefill, memory limit or preemption.\n    mode : str, default \'continuous\'\n        continuous refills each tick; static refills only when its cohort empties.\n\n    Returns\n    -------\n    Schedule\n        Complete finite trace; each tick costs one invented unit, NOT GPU seconds.\n        Empty input has zero makespan and no completions. Residence includes waiting.\n\n    Raises\n    ------\n    ValueError\n        Invalid requests/mode/slots or selector violation.\n    """\n    require_count(slots,\'slots\',1)\n    if mode not in (\'static\',\'continuous\'): raise ValueError(\'Unknown mode\')\n    ids=[r[\'id\'] for r in requests]\n    if len(set(ids))!=len(ids) or any(not isinstance(i,str) or not i for i in ids):\n        raise ValueError(\'Unique nonempty request IDs required\')\n    for r in requests:\n        require_count(r[\'arrival\'],\'arrival\')\n        require_count(r[\'decode_steps\'],\'decode_steps\',1)\n    pending=[dict(r) for r in requests]\n    by_id={r[\'id\']:r for r in requests}\n    remaining={r[\'id\']:r[\'decode_steps\'] for r in requests}\n    active: list[str]=[]\n    ticks: list[Tick]=[]\n    completion: dict[str,int]={}\n    tick=0\n    deadline=sum(r[\'decode_steps\'] for r in requests)+max([0]+[r[\'arrival\'] for r in requests])+1\n    while pending or active:\n        if tick>deadline: raise ValueError(\'Selector made no bounded progress\')\n        if mode==\'continuous\' or not active:\n            selected=admit(pending,tick,slots-len(active))\n            eligible=[r[\'id\'] for r in pending if r[\'arrival\']<=tick]\n            if len(set(selected))!=len(selected) or len(selected)>slots-len(active) or any(i not in eligible for i in selected):\n                raise ValueError(\'Invalid admission\')\n            active.extend(selected)\n            pending=[r for r in pending if r[\'id\'] not in selected]\n        worked=list(active)\n        finished=[]\n        for i in active:\n            remaining[i]-=1\n            if remaining[i]==0: completion[i]=tick+1;finished.append(i)\n        ticks.append(dict(tick=tick,active=worked,\n            waiting=[r[\'id\'] for r in pending if r[\'arrival\']<=tick],finished=finished))\n        active=[i for i in active if i not in finished]\n        tick+=1\n    return dict(mode=mode,ticks=ticks,completion=completion,\n        residence_ticks={i:completion[i]-by_id[i][\'arrival\'] for i in ids},\n        makespan_ticks=tick,idle_slot_ticks=sum(slots-len(t[\'active\']) for t in ticks))\n\n\ndef scalar_attention(keys: Sequence[float], values: Sequence[float],\n                     query: float = 1.0) -> tuple[list[float], float]:\n    """Provided D=1 one-query attention with one GLOBAL normalization.\n\n    Parameters\n    ----------\n    keys : Sequence[float]\n        Nonempty retained scalar keys, all legal for this final causal query.\n    values : Sequence[float]\n        Matching scalar values; keys and values must preserve pairing.\n    query : float, default 1.0\n        Finite scalar query. sqrt(D)=1; no positional rotation in this invented toy.\n\n    Returns\n    -------\n    tuple[list[float], float]\n        Stable global softmax weights and weighted value sum in Python float.\n\n    Raises\n    ------\n    ValueError\n        Empty/mismatched/nonfinite inputs or overflow in query-key scores.\n    """\n    if not keys or len(keys)!=len(values) or any(not math.isfinite(v) for v in [*keys,*values,query]):\n        raise ValueError(\'Finite matching nonempty inputs required\')\n    scores=[query*k for k in keys]\n    if not all(math.isfinite(score) for score in scores):\n        raise ValueError(\'Attention scores overflowed; use a bounded scalar toy\')\n    peak=max(scores)\n    masses=[math.exp(s-peak) for s in scores]\n    weights=[m/sum(masses) for m in masses]\n    return weights,sum(w*v for w,v in zip(weights,values))\n'
HELPER=Path("topic05_scheduling.py")
HELPER.write_text(HELPER_SOURCE)
assert hashlib.sha256(HELPER.read_bytes()).hexdigest()=='8508828a09f3068d548db2e749f473350dd0da3826fdc15959b1efc16b45a1bb'
spec=importlib.util.spec_from_file_location("topic05_scheduling",HELPER)
infra=importlib.util.module_from_spec(spec);spec.loader.exec_module(infra)
from typing import Sequence
KVAccount=infra.KVAccount;Request=infra.Request


In [ ]:
%%lsm
"""Small single-artifact notebook viewers; no inference or state changes."""
from html import escape
from typing import Mapping, Sequence


def format_cell_value(value: object) -> str:
    """Format a scalar or short vector for display without changing stored data.

    Parameters
    ----------
    value : object
        Scalar, list or tuple; floats are shown with six significant digits.
        Nested lists preserve their row boundaries. None denotes a missing value.

    Returns
    -------
    str
        Display text only; original numerical values remain unchanged.

    Raises
    ------
    ValueError
        If a mapping is supplied as one cell. Flatten named metrics into columns
        or separate tables instead of displaying a dense nested result object.
    """
    if value is None:
        return '—'
    if isinstance(value, Mapping):
        raise ValueError('A table cell cannot contain a result dictionary: use named scalar columns')
    if isinstance(value, float):
        return format(value, '.6g')
    if isinstance(value, (list, tuple)):
        return '[' + ', '.join(format_cell_value(item) for item in value) + ']'
    return str(value)


def show_table(
    rows: Sequence[Mapping[str, object]], columns: Sequence[str], caption: str,
) -> None:
    """Display one labelled table with safely escaped cell values.

    Parameters
    ----------
    rows : Sequence[Mapping[str, object]]
        Records for the named columns; missing/None values display an em dash.
        Nested result dictionaries must be decomposed. Display rounding does not
        change the recorded values or calculations.
        An empty population displays an empty table with its caption.
    columns : Sequence[str]
        Display order of fields; headers should carry units where relevant.
    caption : str
        Question/population/evidence scope for this single table.

    Returns
    -------
    None
        Displays one HTML artifact. Does not execute inference or save files.
    """
    from IPython.display import HTML, display
    headers = ''.join('<th scope="col" style="text-align:left;padding:.4em .6em">'
                      + escape(name.replace('_', ' ')) + '&#160;&#160;</th>' for name in columns)
    body = ''.join('<tr>' + ''.join('<td style="vertical-align:top;padding:.4em .6em;'
                    'max-width:32em;overflow-wrap:anywhere;'
                    'border-bottom:1px solid #ccc">' + escape(format_cell_value(row.get(key))) + '&#160;&#160;</td>'
                    for key in columns) + '</tr>' for row in rows)
    display(HTML('<div style="max-width:100%;overflow-x:auto"><table style="border-collapse:collapse">'
                 '<caption style="text-align:left">' + escape(caption)
                 + '<br><small>Display: six significant digits; stored values unchanged.</small>'
                 '</caption><thead><tr>' + headers + '</tr></thead><tbody>'
                 + body + '</tbody></table></div>'))


def show_text(title: str, text: str) -> None:
    """Display one literal input/output or diagnostic text artifact.

    Parameters
    ----------
    title : str
        Case identity and evidence scope.
    text : str
        Literal content; HTML is escaped, not interpreted.

    Returns
    -------
    None
        One readable text panel; no GPU call, write or state change.
    """
    from IPython.display import HTML, display
    display(HTML('<h4>' + escape(title) + '</h4><pre style="white-space:pre-wrap">'
                 + escape(text) + '</pre>'))


## A1. KV payload is not model-weight memory

For Qwen2.5-0.5B-Instruct: 24 full-attention layers, 14 query heads, 2 KV heads, head width 64, FP16 elements 2 bytes. One retained position contains K AND V in every layer. Logical per-layer K and V each have axes [KV head,position,feature]. Explain why using 14 query heads would overcount KV by 7. Predict bytes per position before running. This calculation excludes block tables, alignment and all weights.


In [ ]:
%%lsm
BYTES_PER_POSITION=infra.kv_bytes_per_position(**infra.QWEN)
print("Logical KV payload:",BYTES_PER_POSITION,"bytes/position =",BYTES_PER_POSITION/1024,"KiB/position")


## A2. Round each independent request

Lengths [17,31,33] mean retained positions, not words or future output caps. Block size 16 means 16 positions per block. Requests do not share blocks in this exercise. Predict blocks and unused positions. Why can ceil(sum(lengths)/16) not represent separate block tables? Implement the contract below; no GPU needed.


In [ ]:
%%lsm
def account_kv(lengths: Sequence[int], block_size: int, bytes_per_position: int) -> KVAccount:
    """Account for separately allocated request KV blocks.

    Parameters
    ----------
    lengths : Sequence[int]
        Nonnegative retained-position counts; zero uses no blocks. No sharing.
    block_size : int
        Positive position slots per physical block.
    bytes_per_position : int
        Positive KV payload bytes across all relevant layers.

    Returns
    -------
    KVAccount
        blocks; live_positions; capacity_positions; unused_positions;
        payload_bytes; capacity_bytes. Empty lengths produce all zeros.
        Excludes metadata, allocator reservation and model weights.

    Raises
    ------
    ValueError
        Invalid counts, including bools. Do not mutate lengths.
    """
    raise NotImplementedError('TODO: independent request accounting')


In [ ]:
%%lsm
print("Arithmetic fixture checks:",infra.check_account(account_kv))


In [ ]:
%%lsm
ACCOUNT=account_kv([17,31,33],16,BYTES_PER_POSITION)
import pandas as pd
from IPython.display import display
show_table([{ "quantity":key,"value":value} for key,value in ACCOUNT.items()],["quantity","value"],"Independent request blocks:17/31/33, block16; byte fields are bytes")


### A3. One change: block size

Keep the same lengths and payload dtype. Compare 8, 16, 32. Predict the trade-off: tail slots versus number of table entries. Byte totals below describe ideal block capacity, not measured GPU memory. There is no “best block size” until kernel constraints and real overheads are known.


In [ ]:
%%lsm
BLOCK_SIZES=[8,16,32]
BLOCK_RECORDS=[dict(block_size=b,**account_kv([17,31,33],b,BYTES_PER_POSITION)) for b in BLOCK_SIZES]
show_table(BLOCK_RECORDS,["block_size","blocks","unused_positions","capacity_bytes"],"Same lengths17/31/33; block size in positions, capacity in bytes")


In [ ]:
%%lsm
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(6,3))
ax.bar([str(r['block_size']) for r in BLOCK_RECORDS],[r['unused_positions'] for r in BLOCK_RECORDS])
ax.set(xlabel='Positions per block',ylabel='Unused tail positions',title='Same three invented lengths:17,31,33')
fig.tight_layout();plt.show()


## B. Attention over pages is still one attention

Toy D=1, query 1, keys [1,2,3,4], values [10,20,30,40]. All keys are legal for this final query. Predict whether normalizing each two-key page then averaging the two outputs equals the global result. The helper uses stable softmax. A joint permutation of K/V pairs leaves this full-context sum unchanged. Reading another request’s page or breaking K/V pairing is a different failure.

For this scalar toy, scores are $s_j=qk_j$. One attention is $y=\sum_j e^{s_j}v_j/\sum_j e^{s_j}$, over ALL legal keys. Subtracting the same maximum from every score prevents large exponentials without changing the normalized weights.

For page $p$, define $Z_p=\sum_{j\in p}e^{s_j}$ and $y_p=\sum_{j\in p}e^{s_j}v_j/Z_p$. Then $y=(Z_1y_1+Z_2y_2)/(Z_1+Z_2)$, not $(y_1+y_2)/2$. Here the normalized page masses are 0.119203 and 0.880797: the second page must receive more weight. [Read the worked mechanism in Book](https://levshaazz.github.io/llm-serving-mastery/en/book/05/#attention).


In [ ]:
%%lsm
ATTENTION_WEIGHTS,ATTENTION_OUTPUT=infra.scalar_attention([1,2,3,4],[10,20,30,40])
show_table([dict(position=i,key=i+1,value=10*(i+1),weight=w) for i,w in enumerate(ATTENTION_WEIGHTS)],["position","key","value","weight"],"Invented D=1 final query; one global softmax")


In [ ]:
%%lsm
LOCAL_AVERAGE=(infra.scalar_attention([1,2],[10,20])[1]+infra.scalar_attention([3,4],[30,40])[1])/2
show_table([{ "operation":"global attention","output":ATTENTION_OUTPUT},{"operation":"wrong page-average","output":LOCAL_AVERAGE}],["operation","output"],"Different normalization gives different outputs")


## C1. One short request finishes before its cohort

Two slots. A/B/C arrive at tick 0 with 5/1/1 decode steps; D arrives tick 2 with 2 steps. Each active request executes one decode step per tick. Prefill costs zero ONLY in this toy. A completion at the end of [t,t+1) releases a slot at t+1. Static mode waits for the entire current cohort to empty before refill; continuous mode can refill each tick. Predict C’s completion, then implement arrival-aware FIFO selection. The given lengths create stopping events, not a production oracle for future generated length.


In [ ]:
%%lsm
REQUESTS=infra.REQUESTS
show_table(REQUESTS,["id","arrival","decode_steps"],"Invented requests; time in equal-cost ticks, prefill0")


In [ ]:
%%lsm
def admit_fifo(pending: Sequence[Request], tick: int, free_slots: int) -> list[str]:
    """Select arrived requests without modifying the FIFO pending queue.

    Parameters
    ----------
    pending : Sequence[Request]
        Valid requests already in FIFO order; arrival is an integer toy tick.
    tick : int
        Nonnegative boundary at which admission is evaluated.
    free_slots : int
        Nonnegative available decode slots, not memory blocks or token budget.

    Returns
    -------
    list[str]
        Arrived request IDs in queue order, bounded by free_slots.
        Empty/no-arrived/zero-slot inputs return an empty list.

    Raises
    ------
    ValueError
        Invalid tick or slot count, including bools.
    """
    raise NotImplementedError('TODO: arrival-aware FIFO admission')


In [ ]:
%%lsm
print("Admission fixture checks:",infra.check_admission(admit_fifo))


In [ ]:
%%lsm
STATIC=infra.simulate(admit_fifo,REQUESTS,mode="static")
CONTINUOUS=infra.simulate(admit_fifo,REQUESTS,mode="continuous")


In [ ]:
%%lsm
TRACE_MODE="continuous"
assert TRACE_MODE in ("static","continuous")
TRACE=STATIC if TRACE_MODE=="static" else CONTINUOUS
show_table([{ "tick":r["tick"],"active":",".join(r["active"]),"waiting":",".join(r["waiting"]),"finished_at_next_tick":",".join(r["finished"])} for r in TRACE["ticks"]],["tick","active","waiting","finished_at_next_tick"],"Mode "+TRACE_MODE+"; work during[t,t+1)")


### Read the trace

One row is work during [tick,tick+1). “Finished” refers to the next boundary, not instant completion at row start. Rerun this viewer with static: it must not recompute or time GPU work. Why is B’s early completion compatible with C still waiting in the static cohort? Which assumption makes the plotted tick widths equal?


In [ ]:
%%lsm
show_table([{ "request":r["id"],"arrival_tick":r["arrival"],"static_finish":STATIC["completion"][r["id"]],"continuous_finish":CONTINUOUS["completion"][r["id"]],"static_residence":STATIC["residence_ticks"][r["id"]],"continuous_residence":CONTINUOUS["residence_ticks"][r["id"]]} for r in REQUESTS],["request","arrival_tick","static_residence","continuous_residence"],"All four request residences; subtract each arrival")


In [ ]:
%%lsm
fig,ax=plt.subplots(figsize=(7,3))
for y,(name,result) in enumerate([('static',STATIC),('continuous',CONTINUOUS)]):
    for row in result['ticks']:
        for lane,request in enumerate(row['active']):
            ax.broken_barh([(row['tick'],1)],(y*3+lane,.8),facecolors='#2A6FDB')
            ax.text(row['tick']+.5,y*3+lane+.4,request,ha='center',va='center',color='white')
ax.set(yticks=[.9,3.9],yticklabels=['static','continuous'],xlabel='Invented tick; compact active list, NOT persistent physical lanes',title='Active work, not GPU utilization; prefill omitted')
ax.set_xticks(range(8));fig.tight_layout();plt.show()


## D. Explain before measuring

The real V1 scheduler has a scheduled-token budget, prefill work, KV capacity and possible preemption. “Four clients”, “four active sequences”, “four new tokens” and “four KV blocks” are not interchangeable. A 256-token budget can include several decode positions and a prefill chunk. This toy demonstrates refill logic, not a quantitative performance model of vLLM.

For the real comparison, keep Qwen revision, dtype, exact rendered token IDs, request population, output cap/EOS rule and warmup fixed. Disable prefix caching in the control. Submitted offline burst size1 versus4 is the student change. Pipeline versus vLLM is a whole-stack comparison, not a PagedAttention ablation. Non-streaming cohort wall time cannot establish TTFT or a p95 from three repeats. All failed and semantically wrong requests stay in the denominator.


In [ ]:
%%lsm
EXPLANATION={
 'why_independent_rounding':'TODO: identify the request boundary',
 'why_global_softmax':'TODO: use the two computed outputs',
 'why_ticks_not_seconds':'TODO: name two omitted real costs',
 'what_pipeline_vs_vllm_proves':'TODO: distinguish stack comparison from mechanism causality',
}


## Preserve your work

Keep your modified notebook, block table, both complete traces and explanations. CPU work is not GPU completion. The current code passed its own two complete GPU browser rehearsals with instructor solutions. Do not invent measurements or reuse Topic00’s64client figures as a Topic05 result. Instructor solutions and actual execution captures remain private.


## E. Real model: offline bursts, not HTTP clients

Compare pipeline with vLLM on the same twelve literal tickets, frozen Qwen revision, FP16, natural EOS and output cap 16. Change the submitted burst cohort from 1 to 4. This library comparison does not measure client TTFT or individual E2E latency. It changes more than PagedAttention. Prefix caching is disabled. Three repeats per size/engine retain 144 attempted predictions and 90 cohort intervals. Warmup is outside the measured population. Cohort-size order alternates by repeat.

This experiment does NOT observe a fifth waiting request entering a freed slot: each submitted burst has at most four requests and finishes before the next call. Continuous refill was demonstrated by the toy, not empirically isolated here. Both engines explicitly use repetition penalty 1.0; greedy alone would not override Qwen’s inherited penalty 1.1.

The exact workload passed two standard-RAM T4 browser rehearsals on a paid account. Workload-specific bounds: 6 GiB free host and device before load; 2 GiB free host / 512 MiB free device while working. These do not certify free-account availability or change judge/operator gates. Do not install provider-kernel pip packages. Keep the exact managed runtime.


In [ ]:
%%lsm
#@title Provided readable GPU supervisor; no inference on import { display-mode: "form" }
GPU_HELPER_SOURCE='"""Provided bounded OFFLINE burst comparison; not a PagedAttention ablation.\n\nRun only through the exact browser notebook after CPU prerequisites. Both engines\nreceive the same rendered strings; this measures cohort wall time, NOT TTFT,\nindividual client latency, open-loop capacity or free-account GPU availability.\nImports before explicit child execution are standard-library-only.\n"""\nimport argparse\nimport base64\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport signal\nimport subprocess\nimport sys\nimport time\nimport io\nimport zipfile\nfrom typing import NotRequired, TypedDict\n\nMODEL=\'Qwen/Qwen2.5-0.5B-Instruct\'\nREVISION=\'7ae557604adf67be50417f59c2c2f167def9a775\'\nPROTOCOL=\'topic05-offline-bursts-v1\'\nOUTPUT_POLICY=\'Exclude terminal configured EOS for rate; preserve actual native IDs. Pipeline native tensor includes prompt/padding; vLLM native IDs are completion-only.\'\nGENERATION_PROFILE={\'mode\':\'greedy\',\'repetition_penalty\':1.0,\'presence_penalty\':0.0,\n                    \'frequency_penalty\':0.0,\'output_cap\':16,\'natural_eos\':True,\n                    \'num_sequences\':1,\'seed\':0}\nSYSTEM=\'Classify the support ticket as billing, access, or bug. Output exactly one label and nothing else.\'\nCASES=[\n (\'b1\',\'I was charged twice for my monthly subscription. Please refund the duplicate.\',\'billing\'),\n (\'b2\',\'Please send an invoice for the payment we made last week.\',\'billing\'),\n (\'b3\',\'My credit card expired and the subscription payment was declined.\',\'billing\'),\n (\'b4\',\'I cancelled the subscription but a new charge appeared on my bank statement.\',\'billing\'),\n (\'a1\',\'I forgot my password and cannot sign in to my account.\',\'access\'),\n (\'a2\',\'Our new employee needs permission to open the team workspace.\',\'access\'),\n (\'a3\',\'I replaced my phone and no longer have my two-factor login codes.\',\'access\'),\n (\'a4\',\'My account is locked after too many failed sign-in attempts.\',\'access\'),\n (\'g1\',\'I can log in, but clicking Export crashes the application.\',\'bug\'),\n (\'g2\',\'After successful login, the Save button does nothing and my edits disappear.\',\'bug\'),\n (\'g3\',\'I am signed in. The search page shows an error for every search query.\',\'bug\'),\n (\'g4\',\'My account works, but every image I upload is displayed upside down.\',\'bug\'),\n]\n\n\nclass Headroom(TypedDict):\n    """Observed host/device resource sample; not an OS memory sandbox."""\n    host_available_bytes: int\n    gpu_free_mib: int\n    gpu_used_mib: int\n\n\nclass GeneratedSequence(TypedDict):\n    """Actual engine response and selected IDs; native API tensor scopes differ."""\n    selected_ids: list[int]\n    native_return_ids: list[int]\n    finish_reason: str\n    non_stop_id_count: int\n\n\nclass ComparisonReceipt(TypedDict):\n    """Two owned child files and actual resource snapshots, not learner acceptance."""\n    protocol: str\n    model: str\n    revision: str\n    initial: Headroom\n    children: dict[str, str]\n    post: Headroom\n    scope: str\n\n\nclass PromptRecord(TypedDict):\n    """Exact rendered input, separately retained from native generated tensors."""\n    case_id: str\n    text: str\n    input_ids: list[int]\n\n\nclass RequestRecord(TypedDict):\n    """One actual prediction; the full attempted population remains in the score."""\n    case_id: str\n    cohort_size: int\n    repeat: int\n    output_ids: list[int]\n    output: str\n    gold: str\n    native_return_ids: list[int]\n    finish_reason: str\n    non_stop_id_count: int\n    correct: bool\n\n\nclass CohortRecord(TypedDict):\n    """Synchronized library call interval, not an individual client latency."""\n    repeat: int\n    cohort_size: int\n    case_ids: list[str]\n    wall_s: float\n    resource: Headroom\n\n\nclass BackendArtifact(TypedDict):\n    """Literal source-bound child artifact; pending browser acceptance is separate."""\n    protocol: str\n    status: str\n    backend: str\n    model: str\n    revision: str\n    dtype: str\n    prefix_caching: bool\n    cohort_sizes: list[int]\n    repeats: int\n    output_cap: int\n    versions: dict[str, str]\n    device: str\n    prompts: list[PromptRecord]\n    cohorts: list[CohortRecord]\n    requests: list[RequestRecord]\n    initial: Headroom\n    generation_profile: dict[str, object]\n    output_id_policy: str\n    source_sha256: str\n    eos_ids: list[int]\n    pad_token_id: int\n    cache_block_size: NotRequired[int]\n    kv_cache_memory_bytes: NotRequired[int]\n\n\nclass SummaryRecord(TypedDict):\n    """One engine/size/repeat over all twelve attempts, not twelve client clocks."""\n    engine: str\n    cohort_size: int\n    repeat: int\n    attempted: int\n    wall_s: float\n    requests_per_s: float\n    selected_non_stop_ids_per_s: float\n    correct: int\n    invalid: int\n\n\ndef load_pair(directory: Path) -> dict[str, BackendArtifact]:\n    """Reload and validate actual files for viewers without running inference.\n\n    Parameters\n    ----------\n    directory : Path\n        Explicit current attempt directory containing both completed JSONs.\n\n    Returns\n    -------\n    dict[str, BackendArtifact]\n        Two literal validated artifacts with identical rendered input ledgers.\n\n    Raises\n    ------\n    AssertionError\n        Incomplete/corrupt identity, workload or cross-engine input mismatch.\n    """\n    pair={engine:json.loads((directory/(engine+\'.json\')).read_text())\n          for engine in (\'pipeline\',\'vllm\')}\n    for engine,data in pair.items(): validate_backend(data,engine,[1,4],3)\n    assert pair[\'pipeline\'][\'prompts\']==pair[\'vllm\'][\'prompts\']\n    return pair\n\n\ndef summarize(pair: dict[str, BackendArtifact]) -> list[SummaryRecord]:\n    """Reduce all attempts with a declared cohort clock; never drop bad outputs.\n\n    Parameters\n    ----------\n    pair : dict[str, BackendArtifact]\n        Literal validated artifacts, freshly reloaded by load_pair.\n\n    Returns\n    -------\n    list[SummaryRecord]\n        Twelve engine/size/repeat rows. Rate is12/sum(cohort wall seconds), not\n        mean(1/request latency). Invalid labels remain in all12 attempts.\n    """\n    result=[]\n    for engine,data in pair.items():\n        validate_backend(data,engine,[1,4],3)\n        for size in (1,4):\n            for repeat in range(3):\n                rows=[r for r in data[\'requests\'] if r[\'cohort_size\']==size and r[\'repeat\']==repeat]\n                wall=sum(c[\'wall_s\'] for c in data[\'cohorts\'] if c[\'cohort_size\']==size and c[\'repeat\']==repeat)\n                result.append(dict(engine=engine,cohort_size=size,repeat=repeat,attempted=len(rows),\n                    wall_s=wall,requests_per_s=len(rows)/wall,\n                    selected_non_stop_ids_per_s=sum(r[\'non_stop_id_count\'] for r in rows)/wall,\n                    correct=sum(r[\'correct\'] for r in rows),\n                    invalid=sum(r[\'output\'].strip().lower() not in (\'billing\',\'access\',\'bug\') for r in rows)))\n    return result\n\n\ndef archive_attempt(directory: Path) -> str:\n    """Return a bounded download link for only the explicit owned attempt files.\n\n    Parameters\n    ----------\n    directory : Path\n        Current named topic05-* folder. Partial attempts are preserved as partial;\n        no status promotion, model cache, credentials or arbitrary globs.\n\n    Returns\n    -------\n    str\n        HTML data-download anchor plus exact inventory/digests. At most32MiB\n        uncompressed; browser-download behaviour requires actual browser QA.\n\n    Raises\n    ------\n    ValueError\n        Bad attempt name, symlink, missing evidence or excessive file size.\n    """\n    if not directory.is_dir() or not directory.name.startswith(\'topic05-\') or directory.is_symlink():\n        raise ValueError(\'Explicit owned topic05-* attempt directory required\')\n    allowed=(\'pipeline.json\',\'pipeline.log\',\'pipeline.json.partial.tmp\',\n             \'vllm.json\',\'vllm.log\',\'vllm.json.partial.tmp\',\'index.json\',\'worksheet.json\')\n    inventory=[];raw_files=[];total=0\n    for name in allowed:\n        path=directory/name\n        if not path.exists(): continue\n        if path.is_symlink() or not path.is_file(): raise ValueError(\'Reject symlink/non-file evidence\')\n        size=path.stat().st_size;total+=size\n        if total>32*2**20: raise ValueError(\'Evidence exceeds32MiB; retain files individually\')\n        raw=path.read_bytes();raw_files.append((name,raw))\n        inventory.append(dict(name=name,bytes=len(raw),sha256=hashlib.sha256(raw).hexdigest()))\n    if not raw_files: raise ValueError(\'No actual evidence files; nothing to download\')\n    memory=io.BytesIO()\n    with zipfile.ZipFile(memory,\'w\',compression=zipfile.ZIP_DEFLATED) as bundle:\n        for name,raw in raw_files: bundle.writestr(name,raw)\n        bundle.writestr(\'inventory.json\',json.dumps(inventory,indent=2)+\'\\n\')\n    raw=memory.getvalue();encoded=base64.b64encode(raw).decode(\'ascii\')\n    # Validated name is additionally reduced to safe ASCII; no path interpolation.\n    name=\'\'.join(c for c in directory.name if c.isascii() and (c.isalnum() or c==\'-\'))\n    return \'<a download="\'+name+\'.zip" href="data:application/zip;base64,\'+encoded+\'">Download this actual attempt ZIP</a><p>ZIP SHA256 \'+hashlib.sha256(raw).hexdigest()+\'; \'+str(len(raw_files))+\' actual files; partial attempts stay partial.</p>\'\n\n\ndef terminate_owned_group(process: subprocess.Popen) -> None:\n    """Bound cleanup of this exact child session, including exceptional exits.\n\n    Parameters\n    ----------\n    process : subprocess.Popen\n        The process this supervisor created with start_new_session=True. Never\n        an arbitrary PID. Cleanup fits below the bridge\'s ten-second TERM grace.\n\n    Returns\n    -------\n    None\n        Parent reaped and owned process group no longer present.\n\n    Raises\n    ------\n    RuntimeError\n        The owned group still exists after bounded TERM/KILL cleanup. This is\n        not accepted teardown; preserve logs and disconnect the disposable VM.\n    """\n    try:\n        os.killpg(process.pid,signal.SIGTERM)\n    except ProcessLookupError:\n        process.wait(timeout=2)\n        return\n    try:\n        process.wait(timeout=2)\n    except subprocess.TimeoutExpired:\n        pass\n    try:\n        os.killpg(process.pid,signal.SIGKILL)\n    except ProcessLookupError:\n        pass\n    process.wait(timeout=2)\n    for _ in range(20):\n        try:\n            os.killpg(process.pid,0)\n        except ProcessLookupError:\n            return\n        time.sleep(.1)\n    raise RuntimeError(\'Owned child group remains after cleanup; teardown NOT accepted\')\n\n\ndef headroom(startup: bool = False) -> Headroom:\n    """Check provisional small-model safety bounds without allocating GPU tensors.\n\n    Parameters\n    ----------\n    startup : bool, default False\n        Before load require idle GPU and6GiB free device/host. During runs require\n        2GiB free host and512MiB free device. These NEW bounds are provisional,\n        not transferred Topic04 acceptance or weakened operator/judge gates.\n\n    Returns\n    -------\n    Headroom\n        Actual observed available bytes and MiB on device0.\n\n    Raises\n    ------\n    RuntimeError\n        Missing CUDA diagnostics or insufficient/occupied startup resources.\n    """\n    fields={line.split(\':\')[0]:int(line.split()[1])*1024\n            for line in Path(\'/proc/meminfo\').read_text().splitlines() if line.startswith(\'MemAvailable:\')}\n    raw=subprocess.check_output([\'nvidia-smi\',\'--id=0\',\'--query-gpu=memory.free,memory.used\',\n        \'--format=csv,noheader,nounits\'],text=True).strip()\n    free,used=map(int,raw.split(\',\'))\n    record=Headroom(host_available_bytes=fields[\'MemAvailable\'],gpu_free_mib=free,gpu_used_mib=used)\n    if record[\'host_available_bytes\']<(6 if startup else 2)*2**30 or free<(6144 if startup else 512):\n        raise RuntimeError(\'Stop: insufficient provisional headroom \'+repr(record))\n    if startup and used>256: raise RuntimeError(\'Stop: use an otherwise idle owned GPU \'+repr(record))\n    return record\n\n\ndef run_comparison(directory: Path, cohort_sizes: list[int], repeats: int = 3) -> ComparisonReceipt:\n    """Run two isolated sequential owned child processes; retain all partial attempts.\n\n    Parameters\n    ----------\n    directory : Path\n        New exclusive evidence directory. Never an existing run or model cache.\n    cohort_sizes : list[int]\n        Exactly[1,4] required offline burst sizes, not HTTP clients. This changes\n        submitted cohort size; each engine sees the same12cases/repeat.\n    repeats : int, default 3\n        Exactly3 repeated population passes per size and engine. No p95 claim.\n\n    Returns\n    -------\n    ComparisonReceipt\n        Protocol, two child artifact filenames, idle post-device sample and scope.\n        Raw outputs remain in child files, never fabricated after a failure.\n\n    Raises\n    ------\n    ValueError\n        Wrong workload or pre-existing output directory.\n    RuntimeError\n        Resource, child, timeout or cleanup failure; retain logs/partial JSON.\n    """\n    if cohort_sizes!=[1,4] or any(type(n) is not int for n in cohort_sizes) or type(repeats) is not int or repeats!=3:\n        raise ValueError(\'Required integer sizes[1,4], repeats3\')\n    directory.mkdir(parents=True,exist_ok=False)\n    initial=headroom(True)\n    receipts={}\n    first_prompts=None\n    def interrupted(signum: int, frame: object) -> None:\n        """Convert worker TERM into an exception so owned-child finally runs.\n\n        Parameters\n        ----------\n        signum : int\n            SIGTERM delivered by the owned bridge.\n        frame : object\n            Python signal frame, unused.\n\n        Returns\n        -------\n        None\n            Always raises InterruptedError; no new engine starts afterward.\n        """\n        raise InterruptedError(\'Owned supervisor interrupted by signal \'+str(signum))\n\n    previous_term=signal.signal(signal.SIGTERM,interrupted)\n    try:\n      for backend in [\'pipeline\',\'vllm\']:\n        output=directory/(backend+\'.json\')\n        log=directory/(backend+\'.log\')\n        command=[sys.executable,\'-I\',str(Path(__file__).resolve()),\'--child\',backend,\n            \'--output\',str(output.resolve()),\'--cohort-sizes\',*map(str,cohort_sizes),\'--repeats\',str(repeats)]\n        environment=dict(os.environ,VLLM_WORKER_MULTIPROC_METHOD=\'spawn\',VLLM_USAGE_STATS_DO_NOT_TRACK=\'1\',\n            DO_NOT_TRACK=\'1\',TOKENIZERS_PARALLELISM=\'false\')\n        with log.open(\'xb\') as stream:\n            process=subprocess.Popen(command,stdout=stream,stderr=subprocess.STDOUT,\n                env=environment,start_new_session=True)\n            try:\n                code=process.wait(timeout=420)\n            except subprocess.TimeoutExpired:\n                raise RuntimeError(\'Timeout: partial \'+backend+\' evidence/log preserved\')\n            finally:\n                # Includes KeyboardInterrupt, bridge TERM, timeout and read failures.\n                terminate_owned_group(process)\n        if code!=0: raise RuntimeError(\'Child failed: \'+backend+\'; retain its log/partial JSON\')\n        data=json.loads(output.read_text())\n        validate_backend(data,backend,cohort_sizes,repeats)\n        if first_prompts is None: first_prompts=data[\'prompts\']\n        else: assert data[\'prompts\']==first_prompts,\'Cross-engine rendered input ledger mismatch\'\n        headroom(True)\n        receipts[backend]=output.name\n    finally:\n        signal.signal(signal.SIGTERM,previous_term)\n    result=dict(protocol=PROTOCOL,model=MODEL,revision=REVISION,initial=initial,\n        children=receipts,post=headroom(True),scope=\'offline cohort wall comparison; no TTFT, client latency, mechanism ablation, learner or judge acceptance\')\n    (directory/\'index.json\').write_text(json.dumps(result,indent=2)+\'\\n\')\n    return result\n\n\ndef validate_backend(data: BackendArtifact, backend: str,\n                     sizes: list[int], repeats: int) -> None:\n    """Reject incomplete or wrongly labelled artifacts; does not award browser acceptance.\n\n    Parameters\n    ----------\n    data : BackendArtifact\n        Literal child result, with runtime/ledger/cohorts/request rows.\n    backend : str\n        Required engine label, pipeline or vllm.\n    sizes : list[int]\n        Required[1,4] cohort sizes.\n    repeats : int\n        Required3 repeated full populations.\n\n    Returns\n    -------\n    None\n        Checks identity, finite clocks, complete unique attempts and all-attempt\n        score consistency. This is artifact integrity, not model quality.\n    """\n    assert backend in (\'pipeline\',\'vllm\') and sizes==[1,4] and all(type(n) is int for n in sizes)\n    assert type(repeats) is int and repeats==3\n    assert data[\'protocol\']==PROTOCOL and data[\'status\']==\'complete\' and data[\'backend\']==backend\n    assert data[\'model\']==MODEL and data[\'revision\']==REVISION\n    assert data[\'cohort_sizes\']==sizes and all(type(n) is int for n in data[\'cohort_sizes\'])\n    assert type(data[\'repeats\']) is int and data[\'repeats\']==repeats\n    assert type(data[\'output_cap\']) is int and data[\'output_cap\']==16\n    assert data[\'source_sha256\']==hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\n    assert data[\'dtype\']==\'float16\' and data[\'prefix_caching\'] is False\n    assert data[\'generation_profile\']==GENERATION_PROFILE and data[\'output_id_policy\']==OUTPUT_POLICY\n    assert data[\'versions\'][\'python\']==\'3.11.17\'\n    assert data[\'versions\'][\'transformers\']==\'5.17.0\' and data[\'versions\'][\'torch\']==\'2.13.0+cu129\'\n    assert isinstance(data[\'device\'],str) and data[\'device\'].strip()\n    if backend==\'vllm\':\n        assert data[\'versions\'][\'vllm_distribution\']==\'0.30.0+cu129\' and data[\'versions\'][\'vllm_module\']==\'0.30.0\'\n        assert type(data[\'cache_block_size\']) is int and data[\'cache_block_size\']>0\n        assert data[\'kv_cache_memory_bytes\']==256*2**20\n    def ids(values: list[int]) -> None:\n        """Reject malformed native IDs without importing a tokenizer.\n\n        Parameters\n        ----------\n        values : list[int]\n            Literal IDs. Booleans, strings and negative IDs are rejected.\n\n        Returns\n        -------\n        None\n            Validates structure only, not model vocabulary or decoded text.\n        """\n        assert isinstance(values,list) and all(type(n) is int and n>=0 for n in values)\n    ids(data[\'eos_ids\']);assert sorted(set(data[\'eos_ids\']))==[151643,151645]\n    assert data[\'pad_token_id\']==151645 and type(data[\'pad_token_id\']) is int\n    prompts=data[\'prompts\'];assert isinstance(prompts,list) and len(prompts)==len(CASES)\n    assert [p[\'case_id\'] for p in prompts]==[c[0] for c in CASES]\n    for p,case in zip(prompts,CASES,strict=True):\n        ids(p[\'input_ids\']);assert p[\'input_ids\'] and len(p[\'input_ids\'])+16<=512\n        assert isinstance(p[\'text\'],str) and SYSTEM in p[\'text\'] and case[1] in p[\'text\']\n    ledger={p[\'case_id\']:p for p in prompts}\n    rows=data[\'requests\']; expected={(c[0],s,r) for c in CASES for s in sizes for r in range(repeats)}\n    assert isinstance(rows,list) and len(rows)==len(expected) and {(x[\'case_id\'],x[\'cohort_size\'],x[\'repeat\']) for x in rows}==expected\n    for row in rows:\n        assert type(row[\'cohort_size\']) is int and type(row[\'repeat\']) is int\n        ids(row[\'output_ids\']);ids(row[\'native_return_ids\']);assert len(row[\'output_ids\'])<=16\n        assert not any(n in data[\'eos_ids\'] for n in row[\'output_ids\'][:-1]),\'Selected IDs cannot continue after an EOS\'\n        assert isinstance(row[\'output\'],str) and type(row[\'correct\']) is bool\n        assert row[\'finish_reason\'] in (\'stop\',\'length\')\n        assert type(row[\'non_stop_id_count\']) is int\n        assert row[\'non_stop_id_count\']==sum(n not in data[\'eos_ids\'] for n in row[\'output_ids\'])\n        if row[\'finish_reason\']==\'length\': assert len(row[\'output_ids\'])==16 and not any(n in data[\'eos_ids\'] for n in row[\'output_ids\'])\n        if backend==\'vllm\': assert row[\'native_return_ids\']==row[\'output_ids\']\n        else:\n            native=row[\'native_return_ids\'];prompt=ledger[row[\'case_id\']][\'input_ids\'];padding=0\n            while native[padding:padding+len(prompt)]!=prompt:\n                assert padding<len(native) and native[padding]==data[\'pad_token_id\'];padding+=1\n            tail=native[padding+len(prompt):];stop=next((j for j,n in enumerate(tail) if n in data[\'eos_ids\']),None)\n            selected=tail if stop is None else tail[:stop+1]\n            assert selected==row[\'output_ids\'] and len(tail)<=16\n            assert row[\'finish_reason\']==(\'length\' if stop is None else \'stop\')\n        gold=next(c[2] for c in CASES if c[0]==row[\'case_id\'])\n        assert row[\'gold\']==gold and row[\'correct\']==(row[\'output\'].strip().lower()==gold)\n    expected_cohorts=[(r,s,tuple(c[0] for c in CASES[first:first+s])) for r in range(repeats)\n                     for s in (sizes if r%2==0 else list(reversed(sizes))) for first in range(0,len(CASES),s)]\n    cohorts=data[\'cohorts\'];assert isinstance(cohorts,list) and len(cohorts)==len(expected_cohorts)\n    assert [(c[\'repeat\'],c[\'cohort_size\'],tuple(c[\'case_ids\'])) for c in cohorts]==expected_cohorts\n    for record in cohorts:\n        assert type(record[\'repeat\']) is int and type(record[\'cohort_size\']) is int\n        assert type(record[\'wall_s\']) in (int,float) and math.isfinite(record[\'wall_s\']) and record[\'wall_s\']>0\n        resource=record[\'resource\']\n        assert all(type(resource[k]) is int and resource[k]>=0 for k in Headroom.__annotations__)\n        assert resource[\'host_available_bytes\']>=2*2**30 and resource[\'gpu_free_mib\']>=512\n    initial=data[\'initial\'];assert all(type(initial[k]) is int and initial[k]>=0 for k in Headroom.__annotations__)\n    assert initial[\'host_available_bytes\']>=6*2**30 and initial[\'gpu_free_mib\']>=6144 and initial[\'gpu_used_mib\']<=256\n\n\ndef child(backend: str, output: Path, sizes: list[int], repeats: int) -> None:\n    """Execute an actual child ONLY after the notebook\'s explicit GPU opt-in.\n\n    Parameters\n    ----------\n    backend : str\n        pipeline or vllm. GPU packages are imported here, not by CPU helper import.\n    output : Path\n        Exclusive result path; checkpoint after each cohort, retain failures.\n    sizes : list[int]\n        Required cohort sizes[1,4].\n    repeats : int\n        Required3 passes, interleaved by repeat then size.\n\n    Returns\n    -------\n    None\n        Writes actual raw token/output/clock records. No teacher solution code.\n    """\n    if sizes!=[1,4] or any(type(n) is not int for n in sizes) or type(repeats) is not int or repeats!=3 or output.exists():\n        raise ValueError(\'Wrong/existing attempt\')\n    resource=headroom(True)\n    import torch\n    from importlib.metadata import version\n    from transformers import AutoTokenizer, GenerationConfig\n    assert version(\'transformers\')==\'5.17.0\'\n    assert version(\'torch\')==\'2.13.0+cu129\' and sys.version.split()[0]==\'3.11.17\'\n    tokenizer=AutoTokenizer.from_pretrained(MODEL,revision=REVISION,trust_remote_code=False)\n    tokenizer.padding_side=\'left\'\n    tokenizer.pad_token=tokenizer.eos_token\n    eos_ids=set(tokenizer.encode(tokenizer.eos_token,add_special_tokens=False))\n    generation_config=GenerationConfig.from_pretrained(MODEL,revision=REVISION)\n    configured_eos=generation_config.eos_token_id\n    eos_ids.update(configured_eos if isinstance(configured_eos,list) else [configured_eos])\n    prompts=[tokenizer.apply_chat_template([{\'role\':\'system\',\'content\':SYSTEM},{\'role\':\'user\',\'content\':text}],\n        tokenize=False,add_generation_prompt=True) for _,text,_ in CASES]\n    ledger=[dict(case_id=case[0],text=prompt,input_ids=tokenizer(prompt,add_special_tokens=False)[\'input_ids\'])\n            for case,prompt in zip(CASES,prompts)]\n    assert all(len(p[\'input_ids\'])+16<=512 for p in ledger)\n    identity=dict(protocol=PROTOCOL,status=\'partial\',backend=backend,model=MODEL,revision=REVISION,\n        dtype=\'float16\',prefix_caching=False,cohort_sizes=sizes,repeats=repeats,output_cap=16,\n        versions={\'python\':sys.version.split()[0],\'transformers\':version(\'transformers\'),\'torch\':version(\'torch\')},\n        device=torch.cuda.get_device_name(0),prompts=ledger,cohorts=[],requests=[],initial=resource)\n    identity[\'generation_profile\']=dict(GENERATION_PROFILE)\n    identity[\'source_sha256\']=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\n    identity[\'pad_token_id\']=tokenizer.pad_token_id\n    def save() -> None:\n        """Checkpoint actual child evidence without replacing a different attempt.\n\n        Returns\n        -------\n        None\n            Rewrites only this child\'s explicitly owned partial result file.\n        """\n        temporary=output.with_suffix(\'.json.partial.tmp\')\n        with temporary.open(\'w\',encoding=\'utf8\') as stream:\n            stream.write(json.dumps(identity,ensure_ascii=False,indent=2)+\'\\n\')\n            stream.flush();os.fsync(stream.fileno())\n        os.replace(temporary,output) # Atomic within this exact owned attempt.\n    save()\n    if backend==\'pipeline\':\n        from transformers import AutoModelForCausalLM, pipeline\n        # Single-GPU placement uses ordinary PyTorch .to(), not device_map:\n        # the frozen serving profile deliberately contains no Accelerate.\n        model=AutoModelForCausalLM.from_pretrained(MODEL,revision=REVISION,dtype=torch.float16,\n            trust_remote_code=False,attn_implementation=\'sdpa\').to(\'cuda:0\')\n        assert model.device.type==\'cuda\' and model.dtype==torch.float16\n        engine=pipeline(\'text-generation\',model=model,tokenizer=tokenizer)\n        configured_eos=model.generation_config.eos_token_id\n        eos_ids.update(configured_eos if isinstance(configured_eos,list) else [configured_eos])\n    else:\n        import vllm\n        from vllm import LLM, SamplingParams\n        assert version(\'vllm\')==\'0.30.0+cu129\' and vllm.__version__==\'0.30.0\'\n        identity[\'versions\'].update(vllm_distribution=version(\'vllm\'),vllm_module=vllm.__version__)\n        engine=LLM(model=MODEL,revision=REVISION,tokenizer_revision=REVISION,dtype=\'float16\',\n            max_model_len=512,max_num_seqs=4,max_num_batched_tokens=256,\n            kv_cache_memory_bytes=256*2**20,enable_prefix_caching=False,\n            enable_chunked_prefill=True,enforce_eager=True,trust_remote_code=False)\n        parameters=SamplingParams(temperature=0,max_tokens=16,seed=0,repetition_penalty=1.0,\n            presence_penalty=0.0,frequency_penalty=0.0)\n        config=engine.llm_engine.vllm_config\n        assert config.cache_config.enable_prefix_caching is False\n        assert config.scheduler_config.max_num_seqs==4\n        assert config.scheduler_config.max_num_batched_tokens==256\n        assert config.model_config.max_model_len==512\n        identity[\'cache_block_size\']=config.cache_config.block_size\n        identity[\'kv_cache_memory_bytes\']=config.cache_config.kv_cache_memory_bytes\n    eos_ids.discard(None)\n    identity[\'eos_ids\']=sorted(eos_ids)\n    identity[\'output_id_policy\']=OUTPUT_POLICY\n    def generate(indices: list[int]) -> tuple[list[GeneratedSequence], float]:\n        """Time one actual offline cohort; retain every generated-ID list.\n\n        Parameters\n        ----------\n        indices : list[int]\n            Case indices of one nonempty burst cohort in declared order.\n\n        Returns\n        -------\n        tuple[list[GeneratedSequence], float]\n            Actual selected/native IDs, stopping reason and non-stop count per\n            case, plus synchronized library wall seconds. Native API scopes\n            differ; do not compare full pipeline tensors with vLLM output IDs.\n            No individual client clocks or first-token arrival are observable here.\n        """\n        inputs=[prompts[i] for i in indices]\n        headroom()\n        torch.cuda.synchronize();start=time.perf_counter()\n        if backend==\'pipeline\':\n            raw=engine(inputs,batch_size=len(indices),max_new_tokens=16,do_sample=False,\n                repetition_penalty=1.0,num_beams=1,num_return_sequences=1,\n                return_tensors=True,pad_token_id=tokenizer.pad_token_id,add_special_tokens=False)\n        else: raw=engine.generate(inputs,parameters,use_tqdm=False)\n        torch.cuda.synchronize();elapsed=time.perf_counter()-start\n        outputs=[]\n        for i,record in zip(indices,raw):\n            if backend==\'pipeline\':\n                ids=record[0][\'generated_token_ids\']\n                if hasattr(ids,\'tolist\'): ids=ids.tolist()\n                native=list(ids)\n                expected=ledger[i][\'input_ids\']\n                padding=0\n                while ids[padding:padding+len(expected)]!=expected:\n                    assert padding<len(ids) and ids[padding]==tokenizer.pad_token_id,\'Unexpected pipeline input prefix\'\n                    padding+=1\n                ids=ids[padding+len(expected):]\n                stop=next((j for j,n in enumerate(ids) if n in eos_ids),None)\n                if stop is not None: ids=ids[:stop+1]\n                finish=\'stop\' if stop is not None else \'length\'\n            else:\n                assert list(record.prompt_token_ids)==ledger[i][\'input_ids\'],\'Wrong vLLM rendered input IDs\'\n                ids=list(record.outputs[0].token_ids)\n                native=list(ids);finish=record.outputs[0].finish_reason\n            outputs.append(GeneratedSequence(selected_ids=[int(n) for n in ids],\n                native_return_ids=[int(n) for n in native],finish_reason=finish,\n                non_stop_id_count=sum(n not in eos_ids for n in ids)))\n        assert len(outputs)==len(indices)\n        return outputs,elapsed\n    # Warmup outside each clock/population; the actual paths remain observable.\n    for size in sizes: generate(list(range(size)))\n    for repeat in range(repeats):\n        for size in (sizes if repeat%2==0 else list(reversed(sizes))):\n            for first in range(0,len(CASES),size):\n                indices=list(range(first,first+size))\n                sequences,elapsed=generate(indices)\n                identity[\'cohorts\'].append(dict(repeat=repeat,cohort_size=size,\n                    case_ids=[CASES[i][0] for i in indices],wall_s=elapsed,\n                    resource=headroom()))\n                for i,sequence in zip(indices,sequences):\n                    tokens=sequence[\'selected_ids\']\n                    text=tokenizer.decode(tokens,skip_special_tokens=True)\n                    identity[\'requests\'].append(dict(case_id=CASES[i][0],cohort_size=size,\n                        repeat=repeat,output_ids=tokens,output=text,gold=CASES[i][2],\n                        native_return_ids=sequence[\'native_return_ids\'],finish_reason=sequence[\'finish_reason\'],\n                        non_stop_id_count=sequence[\'non_stop_id_count\'],\n                        correct=text.strip().lower()==CASES[i][2]))\n                save()\n    identity[\'status\']=\'complete\';validate_backend(identity,backend,sizes,repeats);save()\n    # GPU process termination is supervised by the parent; no in-process reuse.\n\n\ndef main() -> None:\n    """Explicit child CLI; never run as a CPU import side effect.\n\n    Returns\n    -------\n    None\n        Executes only with the supplied --child argument and owned output path.\n    """\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--child\',choices=[\'pipeline\',\'vllm\'],required=True)\n    parser.add_argument(\'--output\',type=Path,required=True)\n    parser.add_argument(\'--cohort-sizes\',type=int,nargs=\'+\',required=True)\n    parser.add_argument(\'--repeats\',type=int,required=True)\n    a=parser.parse_args();child(a.child,a.output,a.cohort_sizes,a.repeats)\n\n\nif __name__==\'__main__\': main()\n'
GPU_HELPER=Path("topic05_gpu.py");GPU_HELPER.write_text(GPU_HELPER_SOURCE)
assert hashlib.sha256(GPU_HELPER.read_bytes()).hexdigest()=='eaabc08918229d95606a8429ad4a2757706e6ea67e09620a271f438210d42381'
gpu_spec=importlib.util.spec_from_file_location("topic05_gpu",GPU_HELPER)
gpu=importlib.util.module_from_spec(gpu_spec);gpu_spec.loader.exec_module(gpu)


In [ ]:
%%lsm
show_table([{'case':case,'ticket':text,'gold':gold} for case,text,gold in gpu.CASES],
    ['case','ticket','gold'],'Twelve literal ML inputs; balanced gold labels, not promised predictions')


### Predict before installing GPU packages

Write a falsifiable expectation: which engine/size might process the same twelve tickets faster, and why? Predict whether literal labels must be identical. State one observation that would contradict your prediction. These are hypotheses, not measured answers. In quality, also state a concrete pre-run classroom acceptance/refusal rule using correct/12, invalid/12 and the pipeline control. Do not change that rule after seeing results. Passing this small screen never authorizes a production recommendation. Save the notebook before installing or running inference. The prediction variable is not automatically locked; do not rewrite it after observing results.


In [ ]:
%%lsm
PREDICTION={
 'rate':'TODO: predict the direction and mechanism, not a fabricated number',
 'quality':'TODO: can equal greedy settings still disagree?',
 'falsifier':'TODO: name an observation that would contradict your expectation',
}


In [ ]:
%%lsm
COHORT_SIZES=[1]  # TODO: retain1 and add4; no other workload factor changes.
REPEATS=3
RUN_GPU=False  # Set True only after explaining the CPU results and reading stops.


In [ ]:
%%lsm
def verify_ready() -> gpu.Headroom:
    """Recheck current student work immediately before installation or inference.

    Returns
    -------
    Headroom
        Fresh provisional resource sample after all current prerequisites pass.

    Raises
    ------
    AssertionError
        Missing, invalid or still-TODO explanations, prediction or workload.
    RuntimeError
        Insufficient resources. No installation or model inference starts.
    """
    infra.check_account(account_kv);infra.check_admission(admit_fifo)
    assert COHORT_SIZES==[1,4] and all(type(n) is int for n in COHORT_SIZES)
    assert set(EXPLANATION)=={'why_independent_rounding','why_global_softmax','why_ticks_not_seconds','what_pipeline_vs_vllm_proves'}
    assert set(PREDICTION)=={'rate','quality','falsifier'}
    assert all(isinstance(v,str) and v.strip() and 'TODO' not in v.upper()
               for v in [*EXPLANATION.values(),*PREDICTION.values()])
    return gpu.headroom(True)

if RUN_GPU:
    print('Observed pre-install headroom:',verify_ready())
else:
    print('GPU is OFF. CPU work is not fresh GPU evidence.')


In [ ]:
opt_in=bridge.execute("print(RUN_GPU)")
assert opt_in['ok'] and opt_in['stdout'].strip() in ('True','False')
if opt_in['stdout'].strip()=='True':
    prereq=bridge.execute("verify_ready()")
    assert prereq['ok'], 'Stop: current CPU/prediction/resource gate failed; '+prereq.get('error','')+' '+prereq.get('stderr','')
    RUNTIME_PYTHON=runtime.install_runtime(RUNTIME_ROOT,'serving')
    print('Frozen GPU serving extras installed in managed worker environment')
else:
    print('GPU is OFF; no GPU packages installed')


In [ ]:
%%lsm
from datetime import datetime,timezone
GPU_ATTEMPT=None
if RUN_GPU:
    verify_ready()
    GPU_ATTEMPT=Path('evidence')/datetime.now(timezone.utc).strftime('topic05-%Y%m%dT%H%M%S%fZ')
    GPU_RECEIPT=gpu.run_comparison(GPU_ATTEMPT,COHORT_SIZES,REPEATS)
    print('Two actual owned children completed; preserve',str(GPU_ATTEMPT))
else:
    print('Not run: no new GPU artifact exists')


### What to inspect in the preserved files

Each engine keeps its rendered token ledger, 45 cohort intervals and 72 individual prediction rows. Across both engines this is 90 intervals and 144 predictions. All failures remain partial, never reconstructed as complete. Pipeline native tensors can include prompt/padding; the helper preserves them separately from trimmed selected output IDs. Rate accounting excludes terminal configured EOS, with actual API differences retained. Empty completion is a wrong/invalid task response, not an excuse to drop an attempted request.

Compare each case_id, rendered text and input_ids in the two prompt ledgers, not just the visible ticket. Inspect model revision, dtype and generation_profile (greedy, repetition penalty, output cap and EOS). Pipeline native full tensors are not completion IDs; check the declared native-output scope before counting work.


In [ ]:
%%lsm
import json
GPU_VIEW_ENGINE='vllm'
GPU_VIEW_SIZE=4
GPU_VIEW_REPEAT=0
assert GPU_VIEW_ENGINE in ('pipeline','vllm')
if GPU_ATTEMPT is not None:
    assert type(GPU_VIEW_SIZE) is int and GPU_VIEW_SIZE in COHORT_SIZES
    assert type(GPU_VIEW_REPEAT) is int and GPU_VIEW_REPEAT in range(REPEATS)
    GPU_DATA=json.loads((GPU_ATTEMPT/(GPU_VIEW_ENGINE+'.json')).read_text())
    gpu.validate_backend(GPU_DATA,GPU_VIEW_ENGINE,COHORT_SIZES,REPEATS)
    show_table([{'case_ids':','.join(r['case_ids']),'cohort_wall_ms':r['wall_s']*1000}
        for r in GPU_DATA['cohorts'] if r['cohort_size']==GPU_VIEW_SIZE and r['repeat']==GPU_VIEW_REPEAT],
        ['case_ids','cohort_wall_ms'],'Actual '+GPU_VIEW_ENGINE+' burst'+str(GPU_VIEW_SIZE)+', repeat'+str(GPU_VIEW_REPEAT)+'; all90intervals retained in both files')
else:
    print('No new cohort clocks to display')


In [ ]:
%%lsm
import json
if GPU_ATTEMPT is not None:
    assert GPU_VIEW_ENGINE in ('pipeline','vllm')
    assert type(GPU_VIEW_SIZE) is int and GPU_VIEW_SIZE in COHORT_SIZES
    assert type(GPU_VIEW_REPEAT) is int and GPU_VIEW_REPEAT in range(REPEATS)
    OUTPUT_DATA=json.loads((GPU_ATTEMPT/(GPU_VIEW_ENGINE+'.json')).read_text())
    gpu.validate_backend(OUTPUT_DATA,GPU_VIEW_ENGINE,COHORT_SIZES,REPEATS)
    show_table([{'case':r['case_id'],'gold':r['gold'],'literal_output':r['output'],
        'correct':r['correct']} for r in OUTPUT_DATA['requests']
        if r['cohort_size']==GPU_VIEW_SIZE and r['repeat']==GPU_VIEW_REPEAT],
        ['case','gold','literal_output','correct'],'Actual '+GPU_VIEW_ENGINE+'; twelve literal outputs for selected size/repeat; all attempts retained')
else:
    print('No fresh model outputs to inspect')


### Compare a fixed ML population, not convenient clocks

For each engine/size/repeat compute12 divided by the sum of its cohort wall seconds. The provided reducer does this; do not write timing statistics. Read all three repetitions separately. Compare correct/12 and invalid/12 before recommending a faster route. This tiny repeated population cannot certify generalization or p95. A failed/partial engine has no complete rate.


In [ ]:
%%lsm
if GPU_ATTEMPT is not None:
    show_table(gpu.summarize(gpu.load_pair(GPU_ATTEMPT)),
        ['engine','cohort_size','repeat','attempted','wall_s','requests_per_s','correct','invalid'],
        'All twelve attempts per row; rate12/sum(cohort clocks); three repeated populations, not independent datasets')
else:
    print('No fresh complete pair to summarize')


In [ ]:
%%lsm
if GPU_ATTEMPT is not None:
    RATE_ROWS=gpu.summarize(gpu.load_pair(GPU_ATTEMPT))
    fig,ax=plt.subplots(figsize=(7,3))
    for i,(engine,size) in enumerate([('pipeline',1),('pipeline',4),('vllm',1),('vllm',4)]):
        rates=[r['requests_per_s'] for r in RATE_ROWS if r['engine']==engine and r['cohort_size']==size]
        ax.scatter([i]*3,rates,color='#2A6FDB',s=40)
    ax.set(xticks=range(4),xticklabels=['pipeline1','pipeline4','vLLM1','vLLM4'],
        ylabel='Attempts / synchronized library second',title='Actual three repeat rates; output correctness checked separately')
    ax.set_ylim(bottom=0);fig.tight_layout();plt.show()
else:
    print('No fresh GPU rates to plot')


In [ ]:
%%lsm
import numpy as np
if GPU_ATTEMPT is not None:
    assert type(GPU_VIEW_SIZE) is int and GPU_VIEW_SIZE in COHORT_SIZES
    assert type(GPU_VIEW_REPEAT) is int and GPU_VIEW_REPEAT in range(REPEATS)
    QUALITY_PAIR=gpu.load_pair(GPU_ATTEMPT)
    labels=['billing','access','bug'];predicted=labels+['invalid']
    fig,axes=plt.subplots(1,2,figsize=(9,3.4))
    for ax,(engine,data) in zip(axes,QUALITY_PAIR.items()):
        matrix=np.zeros((3,4),dtype=int)
        for row in data['requests']:
            if row['cohort_size']==GPU_VIEW_SIZE and row['repeat']==GPU_VIEW_REPEAT:
                guess=row['output'].strip().lower()
                matrix[labels.index(row['gold']),predicted.index(guess) if guess in labels else 3]+=1
        assert int(matrix.sum())==12
        ax.imshow(matrix,cmap='Blues',vmin=0,vmax=4)
        for y in range(3):
            for x in range(4):
                rgb=plt.get_cmap('Blues')(matrix[y,x]/4)[:3]
                linear=[v/12.92 if v<=.04045 else ((v+.055)/1.055)**2.4 for v in rgb]
                luminance=sum(v*w for v,w in zip(linear,(.2126,.7152,.0722)))
                ink='black' if (luminance+.05)/.05 >= 1.05/(luminance+.05) else 'white'
                ax.text(x,y,str(matrix[y,x]),ha='center',va='center',color=ink)
        ax.set(xticks=range(4),xticklabels=predicted,yticks=range(3),yticklabels=labels,
            xlabel='Predicted; invalid remains an attempt',ylabel='Gold',title=engine+';12 actual attempts')
    fig.suptitle('Selected burst'+str(GPU_VIEW_SIZE)+', repeat'+str(GPU_VIEW_REPEAT));fig.tight_layout();plt.show()
else:
    print('No fresh model predictions for a quality matrix')


In [ ]:
%%lsm
if GPU_ATTEMPT is not None:
    assert type(GPU_VIEW_SIZE) is int and GPU_VIEW_SIZE in COHORT_SIZES
    assert type(GPU_VIEW_REPEAT) is int and GPU_VIEW_REPEAT in range(REPEATS)
    DISAGREEMENT_PAIR=gpu.load_pair(GPU_ATTEMPT)
    matching={engine:{r['case_id']:r for r in data['requests']
        if r['cohort_size']==GPU_VIEW_SIZE and r['repeat']==GPU_VIEW_REPEAT}
        for engine,data in DISAGREEMENT_PAIR.items()}
    show_table([{'case':case,'gold':gold,'pipeline':matching['pipeline'][case]['output'],
        'vllm':matching['vllm'][case]['output'],
        'literal_equal':matching['pipeline'][case]['output']==matching['vllm'][case]['output']}
        for case,text,gold in gpu.CASES],['case','gold','pipeline','vllm','literal_equal'],
        'All twelve matched literal outputs; equality is not correctness')
else:
    print('No fresh pair of model outputs to compare')


### Your engineering decision

Use your previously written prediction, all three rates, all-attempt quality and one literal error/disagreement if any. If none exist, inspect a correct matching answer and explicitly report zero errors/disagreements; never invent one. Save the notebook before inference; this notebook does not automatically lock the prediction. Do not rewrite it after observing results. Choose a route for this measured workload or explicitly refuse to recommend one. Do not recommend deployment if quality fails your task requirement. The small balanced corpus is a smoke test, not a held-out production evaluation. A throughput change here cannot be attributed specifically to PagedAttention or continuous refill.


In [ ]:
%%lsm
DECISION={
 'choice':'TODO: route or refuse recommendation for the measured workload',
 'evidence':'TODO: quote actual repeat rates, quality counts and a literal example',
 'alternative_explanation':'TODO: another stack difference, not isolated PagedAttention',
 'limitation':'TODO: state an untested requirement and the next experiment',
}


In [ ]:
%%lsm
import json
if GPU_ATTEMPT is not None:
    assert set(DECISION)=={'choice','evidence','alternative_explanation','limitation'}, 'Complete the four required decision fields'
    assert all(isinstance(v,str) and v.strip() and 'TODO' not in v.upper() for v in DECISION.values())
    (GPU_ATTEMPT/'worksheet.json').write_text(json.dumps({'prediction':PREDICTION,
        'decision':DECISION,'cpu_explanation':EXPLANATION},ensure_ascii=False,indent=2)+'\n')
    print('Saved your worksheet; raw engine evidence unchanged')
else:
    print('No GPU attempt; keep CPU explanations in the modified notebook')


In [ ]:
%%lsm
from IPython.display import HTML,display
if GPU_ATTEMPT is not None:
    display(HTML(gpu.archive_attempt(GPU_ATTEMPT)))
else:
    print('No actual GPU attempt files to download')


Download the actual attempt ZIP and your modified notebook before disconnecting. Preserve any partial attempt separately. Do not export model caches or credentials. Results and instructor solutions stay private. A cohort comparison cannot certify deployment quality, p95, TTFT or learner transfer.


In [ ]:
CLOSE_RUNTIME = False  # Set True only after saving all needed evidence.
if CLOSE_RUNTIME:
    bridge.close()
    assert bridge.process.poll() == 0
    print('Owned worker exited; preserve files before disconnecting the VM')
else:
    print('Worker remains available for viewer cells. Save evidence before final closure.')
